In [15]:
!pip install transformers accelerate bitsandbytes gradio torch sentencepiece datasets huggingface_hub

In [16]:
# ============================================================
# SQL Query Generator using SQLCoder-7B-2 + Gradio Interface
# ============================================================

# Imports ──────────────────────────────────────────
import torch
import gradio as gr
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from datasets import load_dataset
import json
import re


In [17]:
#  Configuration ────────────────────────────────────
MODEL_NAME = "defog/sqlcoder-7b-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_4BIT = torch.cuda.is_available()  # Use 4-bit quantization if GPU available

print(f"Using device: {DEVICE}")
print(f"4-bit quantization: {USE_4BIT}")


Using device: cuda
4-bit quantization: True


In [18]:
def build_schema_from_spider(db_id, dataset):
    """
    Build a CREATE TABLE DDL string from Spider dataset rows.
    Extracts column info from the question/query context.
    """
    # Collect all queries for this db_id
    queries = [
        item["query"]
        for item in dataset["train"]
        if item["db_id"] == db_id
    ]

    # Extract table names mentioned in queries
    table_pattern = re.compile(
        r'\bFROM\s+([a-zA-Z_][a-zA-Z0-9_]*)|'
        r'\bJOIN\s+([a-zA-Z_][a-zA-Z0-9_]*)|'
        r'\bINTO\s+([a-zA-Z_][a-zA-Z0-9_]*)',
        re.IGNORECASE
    )

    col_pattern = re.compile(
        r'\bSELECT\s+(.*?)\s+FROM\b',
        re.IGNORECASE | re.DOTALL
    )

    tables = {}
    for query in queries:
        # Find table names
        for match in table_pattern.finditer(query):
            table_name = match.group(1) or match.group(2) or match.group(3)
            if table_name and table_name.lower() not in ("select", "where", "on"):
                if table_name not in tables:
                    tables[table_name] = set()

        # Find columns per table (basic extraction)
        col_match = col_pattern.search(query)
        if col_match:
            cols_str = col_match.group(1)
            cols = [c.strip() for c in cols_str.split(",")]
            for col in cols:
                # Remove aliases, functions, table prefixes
                col = re.sub(r'\(.*?\)', '', col)       # remove functions
                col = re.sub(r'.*\.', '', col)           # remove table prefix
                col = re.sub(r'\s+AS\s+\w+', '', col, flags=re.IGNORECASE)
                col = col.strip().strip('"').strip("'")
                if col and col != "*" and col.isidentifier():
                    # Add to the most recently seen table
                    if tables:
                        last_table = list(tables.keys())[-1]
                        tables[last_table].add(col)

    if not tables:
        return f"-- No schema could be extracted for '{db_id}'\n-- Please provide the schema manually."

    # Build DDL
    ddl_lines = []
    for table_name, columns in tables.items():
        col_defs = ["    id INTEGER PRIMARY KEY"]  # default PK
        for col in sorted(columns):
            if col.lower() not in ("id", "*"):
                col_defs.append(f"    {col} TEXT")
        ddl = (
            f"CREATE TABLE {table_name} (\n"
            + ",\n".join(col_defs)
            + "\n);\n"
        )
        ddl_lines.append(ddl)

    return "\n".join(ddl_lines)

In [19]:
def load_spider_schemas():
    print("Loading Spider dataset...")
    dataset = load_dataset("xlangai/spider", trust_remote_code=True)

    db_questions = {}
    db_queries   = {}

    for item in dataset["train"]:
        db_id = item["db_id"]
        if db_id not in db_questions:
            db_questions[db_id] = []
            db_queries[db_id]   = []
        if len(db_questions[db_id]) < 5:
            db_questions[db_id].append(item["question"])
            db_queries[db_id].append(item["query"])

    return db_questions, db_queries, dataset  #  return dataset too

In [20]:
#  Predefined Schemas (for demo without full Spider) ─
SAMPLE_SCHEMAS = {
    "ecommerce": """
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100),
    city VARCHAR(50),
    country VARCHAR(50),
    signup_date DATE
);

CREATE TABLE products (
    product_id INT PRIMARY KEY,
    name VARCHAR(100),
    category VARCHAR(50),
    price DECIMAL(10, 2),
    stock_quantity INT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT REFERENCES customers(customer_id),
    order_date DATE,
    total_amount DECIMAL(10, 2),
    status VARCHAR(20)
);

CREATE TABLE order_items (
    item_id INT PRIMARY KEY,
    order_id INT REFERENCES orders(order_id),
    product_id INT REFERENCES products(product_id),
    quantity INT,
    unit_price DECIMAL(10, 2)
);
""",
    "university": """
CREATE TABLE students (
    student_id INT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    major VARCHAR(50),
    gpa DECIMAL(3, 2),
    enrollment_year INT
);

CREATE TABLE courses (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100),
    department VARCHAR(50),
    credits INT,
    instructor VARCHAR(100)
);

CREATE TABLE enrollments (
    enrollment_id INT PRIMARY KEY,
    student_id INT REFERENCES students(student_id),
    course_id INT REFERENCES courses(course_id),
    grade VARCHAR(2),
    semester VARCHAR(20)
);
""",
    "hospital": """
CREATE TABLE patients (
    patient_id INT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    gender VARCHAR(10),
    blood_type VARCHAR(5),
    admission_date DATE
);

CREATE TABLE doctors (
    doctor_id INT PRIMARY KEY,
    name VARCHAR(100),
    specialty VARCHAR(50),
    department VARCHAR(50),
    years_experience INT
);

CREATE TABLE appointments (
    appointment_id INT PRIMARY KEY,
    patient_id INT REFERENCES patients(patient_id),
    doctor_id INT REFERENCES doctors(doctor_id),
    appointment_date DATE,
    diagnosis VARCHAR(200),
    treatment VARCHAR(200)
);

CREATE TABLE medications (
    medication_id INT PRIMARY KEY,
    appointment_id INT REFERENCES appointments(appointment_id),
    drug_name VARCHAR(100),
    dosage VARCHAR(50),
    duration_days INT
);
""",
    "custom": ""  # User will provide their own
}

# Sample questions for each schema
SAMPLE_QUESTIONS = {
    "ecommerce": [
        "Show me the top 5 customers by total order amount",
        "Which products are low in stock (less than 10 units)?",
        "What is the total revenue per product category?",
        "List all orders placed in the last 30 days with customer names",
        "Find customers who have never placed an order",
    ],
    "university": [
        "List all students with GPA above 3.5",
        "Which course has the most enrollments?",
        "Show average GPA per major",
        "Find students who are enrolled in more than 3 courses",
        "List courses taught by each instructor",
    ],
    "hospital": [
        "How many patients were admitted per month this year?",
        "Which doctor has treated the most patients?",
        "List all patients currently on more than 2 medications",
        "Find the most common diagnoses",
        "Show all appointments for patients above 60 years old",
    ],
    "custom": []
}


In [21]:
#Load SQLCoder Model
def load_model():
    """Load SQLCoder-7B-2 with optional 4-bit quantization."""
    print(f"Loading {MODEL_NAME}...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    if USE_4BIT:
        # 4-bit quantization for memory efficiency on GPU
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        # CPU fallback (slower)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            device_map="cpu",
            trust_remote_code=True,
        )

    print("Model loaded successfully!")
    return tokenizer, model

In [22]:
# Prompt Template (SQLCoder format) ─────────────────
def build_prompt(question: str, schema: str, dialect: str = "SQL") -> str:
    """Build the SQLCoder prompt using its expected format."""
    prompt = f"""### Task
Generate a {dialect} query to answer [QUESTION]{question}[/QUESTION]

### Instructions
- If you cannot answer the question with the available database schema, return 'I do not know'
- Remember that revenue is price multiplied by quantity
- Remember that 1 means true and 0 means false in the database
- Ensure all columns referenced are present in the schema

### Database Schema
The query will run on a database with the following schema:
{schema}

### Answer
Given the database schema, here is the SQL query that answers [QUESTION]{question}[/QUESTION]
[SQL]"""
    return prompt


In [23]:
# SQL Generation Function ──────────────────────────
def generate_sql(
    tokenizer,
    model,
    question: str,
    schema: str,
    dialect: str = "SQL",
    max_new_tokens: int = 300,
    temperature: float = 0.1,
    num_beams: int = 4,
) -> str:
    """Generate SQL query from natural language question."""

    if not question.strip():
        return " Please enter a question."
    if not schema.strip():
        return " Please provide a database schema."

    prompt = build_prompt(question, schema, dialect)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            num_beams=num_beams,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens (not the prompt)
    generated = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    # Extract just the SQL part
    sql = generated.strip()

    # Clean up end tokens
    for stop in ["[/SQL]", "###", "\n\n\n"]:
        if stop in sql:
            sql = sql[:sql.index(stop)]

    return sql.strip()

In [24]:
def build_gradio_app(tokenizer, model, spider_db_list, spider_questions, spider_dataset):
    """Build and launch the Gradio interface."""

    def on_schema_select(schema_name):
        schema = SAMPLE_SCHEMAS.get(schema_name, "")
        questions = SAMPLE_QUESTIONS.get(schema_name, [])
        sample_text = "\n".join([f"• {q}" for q in questions]) if questions else ""
        return schema, sample_text

    def on_generate(question, schema_text, dialect, max_tokens, temperature):
        if not question.strip():
            return "Please enter a question.", ""
        if not schema_text.strip():
            return "Please provide or select a schema.", ""
        try:
            sql = generate_sql(
                tokenizer, model,
                question=question,
                schema=schema_text,
                dialect=dialect,
                max_new_tokens=int(max_tokens),
                temperature=float(temperature),
            )
            formatted = f"```sql\n{sql}\n```"
            explanation = f" Generated {dialect} query for: \"{question}\""
            return formatted, explanation
        except Exception as e:
            return f" Error: {str(e)}", ""

    def on_sample_schema_change(name):
        schema = SAMPLE_SCHEMAS.get(name, "")
        qs = SAMPLE_QUESTIONS.get(name, [])
        return schema, "\n".join(f"• {q}" for q in qs)

    #  Replace on_spider_db_select with this:
    def on_spider_db_select(db_id):
        if not db_id:
            return "", ""

        # Get sample questions
        questions = spider_questions.get(db_id, [])
        q_text = "\n".join(f"• {q}" for q in questions)

        # Try SAMPLE_SCHEMAS first (ecommerce/university/hospital)
        if db_id in SAMPLE_SCHEMAS:
            schema = SAMPLE_SCHEMAS[db_id]
        else:
            #  Build schema dynamically from Spider queries
            schema = build_schema_from_spider(db_id, spider_dataset)

        return schema, q_text

    # ── Gradio UI Layout ──────────────────────────────────────
    with gr.Blocks(title="QueryForge — SQL Query Generator", theme=gr.themes.Soft()) as demo:

        # Everything indented inside gr.Blocks()
        gr.Markdown("""
        #  QueryForge
        ### Natural Language → SQL | Powered by SQLCoder-7B-2
        ---
        """)

        with gr.Tabs():

            # ── Tab 1: Sample Schemas ─────────────────────────
            with gr.TabItem(" Sample Schemas"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("###  Settings")

                        schema_selector = gr.Dropdown(
                            choices=list(SAMPLE_SCHEMAS.keys()),
                            value="ecommerce",
                            label=" Select Sample Database",
                        )

                        sample_questions_box = gr.Textbox(
                            label=" Sample Questions",
                            lines=6,
                            interactive=False,
                            value="\n".join([f"• {q}" for q in SAMPLE_QUESTIONS["ecommerce"]])
                        )

                        dialect_1 = gr.Radio(
                            choices=["SQL", "MySQL", "PostgreSQL", "SQLite", "BigQuery"],
                            value="SQL",
                            label=" SQL Dialect"
                        )

                        with gr.Accordion(" Advanced", open=False):
                            max_tokens_1 = gr.Slider(100, 500, value=300, step=50,
                                                     label="Max Tokens")
                            temperature_1 = gr.Slider(0.0, 1.0, value=0.1, step=0.05,
                                                      label="Temperature")

                    with gr.Column(scale=2):
                        gr.Markdown("###  Schema (DDL)")
                        schema_input_1 = gr.Textbox(
                            value=SAMPLE_SCHEMAS["ecommerce"],
                            lines=14,
                            label="CREATE TABLE Statements",
                        )

                        question_input_1 = gr.Textbox(
                            lines=2,
                            label=" Your Question",
                            placeholder="e.g. Show me the top 5 customers by total order amount",
                        )

                        generate_btn_1 = gr.Button(" Generate SQL", variant="primary", size="lg")

                status_1 = gr.Textbox(label="Status", interactive=False, lines=1)
                output_1 = gr.Textbox(label=" Generated SQL", lines=8, interactive=False)

            # ── Tab 2: Spider Dataset Browser ─────────────────
            with gr.TabItem(" Spider Dataset (200+ Databases)"):
                gr.Markdown("""
                ### Browse all 200+ real-world databases from the Spider benchmark dataset.
                Select a database to auto-load its schema and sample questions.
                """)

                with gr.Row():
                    with gr.Column(scale=1):
                        spider_db_dropdown = gr.Dropdown(
                            choices=spider_db_list,
                            value=spider_db_list[0] if spider_db_list else None,
                            label=" Spider Database",
                            info=f"{len(spider_db_list)} databases available",
                            filterable=True,
                        )

                        spider_questions_box = gr.Textbox(
                            label=" Sample Questions from Spider",
                            lines=8,
                            interactive=False,
                        )

                        dialect_2 = gr.Radio(
                            choices=["SQL", "MySQL", "PostgreSQL", "SQLite", "BigQuery"],
                            value="SQL",
                            label=" SQL Dialect"
                        )

                        with gr.Accordion(" Advanced", open=False):
                            max_tokens_2 = gr.Slider(100, 500, value=300, step=50,
                                                     label="Max Tokens")
                            temperature_2 = gr.Slider(0.0, 1.0, value=0.1, step=0.05,
                                                      label="Temperature")

                    with gr.Column(scale=2):
                        gr.Markdown("###  Auto-loaded Schema")
                        schema_display = gr.Textbox(
                            label="Schema DDL (auto-filled from Spider)",
                            lines=14,
                            interactive=True,
                            placeholder="Select a database from the dropdown to load its schema...",
                        )

                        question_input_2 = gr.Textbox(
                            lines=2,
                            label=" Your Question",
                            placeholder="Click a sample question above or type your own...",
                        )

                        with gr.Row():
                            generate_btn_2 = gr.Button(" Generate SQL", variant="primary", size="lg")
                            clear_btn = gr.Button(" Clear", size="lg")

                status_2 = gr.Textbox(label="Status", interactive=False, lines=1)
                output_2 = gr.Textbox(label=" Generated SQL", lines=8, interactive=False)

        # ── Event Handlers (still inside gr.Blocks) ───────────

        # Tab 1 — schema selector
        schema_selector.change(
            fn=on_sample_schema_change,
            inputs=[schema_selector],
            outputs=[schema_input_1, sample_questions_box]
        )

        # Tab 1 — generate button
        generate_btn_1.click(
            fn=on_generate,
            inputs=[question_input_1, schema_input_1, dialect_1, max_tokens_1, temperature_1],
            outputs=[output_1, status_1]
        )

        # Tab 2 — Spider DB dropdown
        spider_db_dropdown.change(
            fn=on_spider_db_select,
            inputs=[spider_db_dropdown],
            outputs=[schema_display, spider_questions_box]
        )

        # Tab 2 — generate button
        generate_btn_2.click(
            fn=on_generate,
            inputs=[question_input_2, schema_display, dialect_2, max_tokens_2, temperature_2],
            outputs=[output_2, status_2]
        )

        # Tab 2 — clear button
        clear_btn.click(
            fn=lambda: ("", "", "", ""),
            outputs=[question_input_2, schema_display, output_2, status_2]
        )

    return demo

In [1]:
if __name__ == "__main__":
    tokenizer, model = load_model()

    spider_questions, spider_queries, spider_dataset = load_spider_schemas()
    spider_db_list = sorted(spider_questions.keys())

    demo = build_gradio_app(
        tokenizer, model,
        spider_db_list,
        spider_questions,
        spider_dataset       #  pass dataset here
    )
    demo.launch(server_name="0.0.0.0", server_port=7860, share=False)